<a href="https://colab.research.google.com/github/cesarpc1307/Quorum-IA-Detector/blob/main/Qu%C3%B3rum_IA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Preparación de librerías
!pip install python-docx transformers torch -q

import docx
import os
from transformers import pipeline

# 2. Inicialización de los 3 Jueces en la GPU T4
print("Iniciando Escuadrón de Jueces en la GPU T4...")
juez_1 = pipeline("text-classification", model="roberta-base-openai-detector", device=0)
juez_2 = pipeline("text-classification", model="Hello-SimpleAI/chatgpt-detector-roberta", device=0)
juez_3 = pipeline("text-classification", model="openai-community/roberta-large-openai-detector", device=0)

print("✅ Los 3 jueces están listos para la deliberación.\n")

def auditoria_final_unah(ruta):
    if not os.path.exists(ruta):
        return print(f"❌ Error: El archivo '{ruta}' no encontrado.")

    doc = docx.Document(ruta)
    parrafos = [p.text.strip() for p in doc.paragraphs if len(p.text.strip()) > 40]

    print(f"--- ANALIZANDO ARTÍCULO: {ruta} ---")
    print(f"Bloques de texto detectados: {len(parrafos)}\n")

    ia_j1, ia_j2, ia_j3 = 0, 0, 0
    conteo_consenso = 0

    try:
        # Inferencia Batch
        res_j1 = juez_1(parrafos, truncation=True, max_length=512)
        res_j2 = juez_2(parrafos, truncation=True, max_length=512)
        res_j3 = juez_3(parrafos, truncation=True, max_length=512)

        for i in range(len(parrafos)):
            # --- Extracción Blindada ---
            d1 = res_j1[i]
            while isinstance(d1, list): d1 = d1
            es_ia_1 = (d1['label'] == 'Fake')
            if es_ia_1: ia_j1 += 1

            d2 = res_j2[i]
            while isinstance(d2, list): d2 = d2
            es_ia_2 = (d2['label'].upper() in ['CHATGPT', 'LABEL_1', 'FAKE'])
            if es_ia_2: ia_j2 += 1

            d3 = res_j3[i]
            while isinstance(d3, list): d3 = d3
            es_ia_3 = (d3['label'] == 'Fake')
            if es_ia_3: ia_j3 += 1

            votos_ia = int(es_ia_1) + int(es_ia_2) + int(es_ia_3)
            if votos_ia >= 2:
                marca, peso = "🔴 [IA - MAYORÍA]", 1
            elif votos_ia == 1:
                marca, peso = "🟡 [DUDA / REVISAR]", 0.5
            else:
                marca, peso = "🟢 [HUMANO]", 0

            conteo_consenso += peso
            print(f"P{i+1:02d}: {marca} (Votos IA: {votos_ia}/3) | {parrafos[i][:60]}...", flush=True)

        # --- INFORMES FINALES ---
        total = len(parrafos)

        # (Aquí se imprimen los 3 resúmenes de los jueces que ya tenías...)
        for j, (nombre, hits, desc) in enumerate([
            ("RoBERTa Base", ia_j1, "Analiza patrones estadísticos y predictibilidad."),
            ("ChatGPT Detector", ia_j2, "Especializado en la huella digital semántica de ChatGPT."),
            ("RoBERTa Large", ia_j3, "Evaluación de contexto profundo con alta precisión.")
        ]):
            print("\n" + "-"*45)
            print(f"⚖️ RESUMEN JUEZ {j+1} ({nombre})")
            print(f"Descripción: {desc}")
            print(f"RESULTADO -> IA: {hits} | Humano: {total-hits} | Prob: {(hits/total)*100:.2f}%")

        # INFORME DE CONSENSO TÉCNICO
        pct_final = (conteo_consenso / total) * 100
        print("\n" + "="*45)
        print(f"📊 INFORME DE CONSENSO TÉCNICO (PROMEDIO)")
        print(f"="*45)
        print(f"Párrafos procesados: {total}")
        print(f"Índice de Probabilidad IA: {pct_final:.2f}%")
        print(f"Veredicto: {'ALTA SOSPECHA' if pct_final > 20 else 'INTEGRIDAD VERIFICADA'}")

        # --- EL SELLO DE AUTORÍA (TU APORTE) ---
        print("\n" + "—"*45)
        print(f"🛠️ Auditoría técnica realizada por: Ing. César Pineda")
        print(f"🇭🇳 Gracias, Lempira, Honduras")
        print(f"📌 Proyecto: Quórum-IA - Verificación de Integridad")
        print("—"*45)

    except Exception as e:
        print(f"❌ Error en el proceso técnico: {e}")

# 3. EJECUTAR
auditoria_final_unah('mi_articulo.docx')